# azure_core: 5 Basic Usage Examples

In [ ]:
# Example 1: Import and check version
import azure.core

print(azure.core.__version__)
assert isinstance(azure.core.__version__, str) and azure.core.__version__[0].isdigit()

In [ ]:
# Example 2: Using azure.core.exceptions
from azure.core.exceptions import AzureError, HttpResponseError, ResourceNotFoundError

try:
    raise AzureError("Test error")
except AzureError as e:
    assert str(e) == "Test error" and e.message == "Test error"
    print(f"Caught AzureError: {e}")

# The HTTP error types share AzureError as their base
assert issubclass(ResourceNotFoundError, HttpResponseError)
assert issubclass(HttpResponseError, AzureError)

In [ ]:
# Example 3: Send requests through a PipelineClient with a canned transport
# RequestsTransport is azure-core's real HTTP transport; CannedAdapter answers its requests
# instead of the network, so the policies and response handling run unchanged.
import http
import io
import json

import requests
from azure.core import PipelineClient
from azure.core.exceptions import AzureError, HttpResponseError, ResourceNotFoundError, map_error
from azure.core.pipeline.policies import HeadersPolicy, RetryPolicy, UserAgentPolicy
from azure.core.pipeline.transport import RequestsTransport
from azure.core.rest import HttpRequest


class CannedAdapter(requests.adapters.BaseAdapter):
    def __init__(self, handler):
        super().__init__()
        self.handler = handler  # request -> (status code, JSON payload)
        self.requests = []

    def send(self, request, **kwargs):
        self.requests.append(request)
        status_code, payload = self.handler(request)
        body = json.dumps(payload).encode()
        response = requests.Response()
        response.status_code = status_code
        response.reason = http.HTTPStatus(status_code).phrase
        response.headers = requests.structures.CaseInsensitiveDict(
            {"Content-Type": "application/json", "Content-Length": str(len(body))})
        response.raw = io.BytesIO(body)
        response.url = request.url
        response.request = request
        return response

    def close(self):
        pass


def items_service(request):
    if request.path_url == "/items/1":
        return 200, {"id": 1, "name": "widget"}
    return 404, {"error": {"code": "NotFound", "message": "no such item"}}


adapter = CannedAdapter(items_service)
session = requests.Session()
session.mount("https://", adapter)
client = PipelineClient(
    base_url="https://example.invalid",
    transport=RequestsTransport(session=session),
    policies=[HeadersPolicy({"x-example": "1"}), UserAgentPolicy(sdk_moniker="notebook-example/1.0"),
              RetryPolicy(retry_total=0)],
)

response = client.send_request(HttpRequest("GET", "https://example.invalid/items/1"))
assert response.status_code == 200
assert response.json() == {"id": 1, "name": "widget"}
sent = adapter.requests[0]
assert sent.headers["x-example"] == "1"
assert "notebook-example/1.0" in sent.headers["User-Agent"]
print("GET /items/1 ->", response.status_code, response.json())

# A 404 surfaces as HttpResponseError, and map_error turns it into the specific error type
missing = client.send_request(HttpRequest("GET", "https://example.invalid/items/2"))
assert missing.status_code == 404
try:
    missing.raise_for_status()
except HttpResponseError as err:
    assert isinstance(err, AzureError) and err.status_code == 404
else:
    raise AssertionError("raise_for_status() did not raise for a 404 response")
try:
    map_error(missing.status_code, missing, {404: ResourceNotFoundError})
except ResourceNotFoundError as err:
    assert err.status_code == 404
    print("GET /items/2 ->", type(err).__name__, err.status_code)
else:
    raise AssertionError("map_error() did not raise ResourceNotFoundError for a 404 response")

In [ ]:
# Example 4: Using azure.core.settings
import logging

from azure.core.settings import settings

# config() returns a snapshot; keyword arguments override individual settings in that snapshot only
snapshot = settings.config(log_level=logging.WARNING)
assert snapshot.log_level == logging.WARNING
assert snapshot.tracing_enabled in (True, False)
print(f"log_level={snapshot.log_level} tracing_enabled={snapshot.tracing_enabled}")

In [ ]:
# Example 5: Using azure.core.tracing
from azure.core.tracing import AbstractSpan


class MySpan(AbstractSpan):
    def __init__(self):
        self.name = "myspan"
        self.attributes = {}

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        pass

    def span_impl(self):
        return self

    def add_attribute(self, key, value):
        self.attributes[key] = value


with MySpan() as span:
    span.add_attribute("foo", "bar")
assert span.attributes == {"foo": "bar"}
print(f"Attributes: {span.attributes}")